In [ ]:
! pip install geopandas -qqq
! pip install shapely -qqq
! pip install rasterio -qqq
! pip install rasterstats -qqq
! pip install pyproj -qqq
! pip install pandas -qqq

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import os
import re
import warnings
import glob
import tempfile
import rasterio
from rasterio.mask import mask as raster_mask
from rasterstats import zonal_stats
from shapely.ops import transform
from pyproj import Transformer

warnings.filterwarnings('ignore')

def clean_municipality_name(name):
    if pd.isna(name) or not isinstance(name, str): return name
    cleaned = re.sub(r'^ru:', '', name, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*\([^)]*\)', '', cleaned)
    return cleaned.strip()

def advanced_grid_pipeline_v11(layers: dict, population_df: pd.DataFrame,
                              output: str, size_m: int = 1000, target_crs: str = None,
                              ntl_path: str = None, relief_folder: str = None):
    print("1. Загрузка и приведение к метрической проекции")
    gdfs = {}
    for name, path in layers.items():
        gdf = gpd.read_file(path)
        if gdf.empty: continue
        if target_crs:
            gdf = gdf.to_crs(target_crs)
        else:
            target_crs = gdf.estimate_utm_crs() if gdf.crs.is_geographic else gdf.crs
            gdf = gdf.to_crs(target_crs)
        gdfs[name] = gdf
    if not gdfs: raise ValueError("Слои пусты.")

    print(f"2. Создание сетки {size_m//1000}x{size_m//1000} км")
    bounds_list = [gdf.total_bounds for gdf in gdfs.values()]
    bounds_array = np.array(bounds_list)
    minx, miny = bounds_array[:, 0].min(), bounds_array[:, 1].min()
    maxx, maxy = bounds_array[:, 2].max(), bounds_array[:, 3].max()
    x = np.arange(np.floor(minx/size_m)*size_m, np.ceil(maxx/size_m)*size_m, size_m)
    y = np.arange(np.floor(miny/size_m)*size_m, np.ceil(maxy/size_m)*size_m, size_m)
    grid_geoms = [box(xi, yi, xi+size_m, yi+size_m) for xi in x for yi in y]
    gdf_grid = gpd.GeoDataFrame({'grid_id': range(len(grid_geoms))}, geometry=grid_geoms, crs=target_crs)
    cell_area = size_m ** 2

    
    admin_union_geom = None
    if 'admin' in gdfs:
        print("️3. Привязка муниципалитетов")
        admin_gdf = gdfs['admin'].copy()
        name_col = 'wikipedia' if 'wikipedia' in admin_gdf.columns else 'name'
        if name_col in admin_gdf.columns:
            admin_gdf['name_clean'] = admin_gdf[name_col].apply(clean_municipality_name)
            inter_a = gpd.overlay(gdf_grid[['grid_id', 'geometry']], admin_gdf[['geometry', 'name_clean']], how='intersection')
            if not inter_a.empty:
                inter_a['inter_area'] = inter_a.geometry.area
                idx_max = inter_a.groupby('grid_id')['inter_area'].idxmax()
                dominant = inter_a.loc[idx_max, ['grid_id', 'name_clean', 'inter_area']].rename(
                    columns={'name_clean': 'municipality', 'inter_area': 'municipality_area'})
                gdf_grid = gdf_grid.merge(dominant, on='grid_id', how='left')
                gdf_grid['municipality_area'] = gdf_grid['municipality_area'].fillna(0.0)
            admin_union_geom = admin_gdf.unary_union

    
    if 'buildings' in gdfs:
        print("4. Расчёт метрик зданий")
        inter_b = gpd.overlay(gdf_grid[['grid_id', 'geometry']], gdfs['buildings'][['geometry']], how='intersection')
        if not inter_b.empty:
            inter_b['area'] = inter_b.geometry.area
            agg_b = inter_b.groupby('grid_id').agg(
                total_building_area=('area', 'sum'),
                number_of_buildings=('area', 'size')
            ).reset_index()
            agg_b['building_coverage'] = agg_b['total_building_area'] / cell_area
            gdf_grid = gdf_grid.merge(agg_b, on='grid_id', how='left')
        else:
            for c in ['total_building_area', 'building_coverage', 'number_of_buildings']: gdf_grid[c] = 0.0

    
    if 'roads' in gdfs:
        print("5. Расчёт метрик дорог")
        roads_gdf = gdfs['roads'][['geometry']].copy()
        roads_gdf = roads_gdf.explode(index_parts=True).reset_index(drop=True)
        roads_gdf = roads_gdf[roads_gdf.geometry.is_valid & ~roads_gdf.geometry.is_empty]
        road_grid_join = gpd.sjoin(roads_gdf, gdf_grid[['grid_id', 'geometry']], how='inner', predicate='intersects')
        if not road_grid_join.empty:
            clipped = gpd.clip(road_grid_join[['geometry', 'grid_id']], gdf_grid)
            clipped = clipped[~clipped.geometry.is_empty]
            if not clipped.empty:
                clipped['length'] = clipped.geometry.length
                agg_r = clipped.groupby('grid_id')['length'].sum().reset_index()
                agg_r.columns = ['grid_id', 'road_length']
                agg_r['road_density'] = agg_r['road_length'] / cell_area
                gdf_grid = gdf_grid.merge(agg_r, on='grid_id', how='left')
        if 'road_length' not in gdf_grid.columns: gdf_grid['road_length'] = 0.0
        if 'road_density' not in gdf_grid.columns: gdf_grid['road_density'] = 0.0

    
    if 'nature' in gdfs:
        print("6. Расчёт метрик природы")
        inter_n = gpd.overlay(gdf_grid[['grid_id', 'geometry']], gdfs['nature'][['geometry']], how='intersection')
        if not inter_n.empty:
            inter_n['area'] = inter_n.geometry.area
            agg_n = inter_n.groupby('grid_id')['area'].sum().reset_index()
            agg_n.columns = ['grid_id', 'nature_area']
            agg_n['nature_density'] = agg_n['nature_area'] / cell_area
            gdf_grid = gdf_grid.merge(agg_n, on='grid_id', how='left')
        else:
            for c in ['nature_area', 'nature_density']: gdf_grid[c] = 0.0

   
    if ntl_path and admin_union_geom:
        print("7. Обработка ночных огней")
        temp_dir = tempfile.mkdtemp()
        cropped_tif = os.path.join(temp_dir, 'ntl_cropped_log.tif')
        with rasterio.open(ntl_path) as src:
            transformer = Transformer.from_crs(gdf_grid.crs, src.crs, always_xy=True)
            admin_geom_transformed = transform(transformer.transform, admin_union_geom)
            out_image, out_transform = raster_mask(src, [admin_geom_transformed], crop=True, nodata=np.nan)
            out_meta = src.meta.copy()
            out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2],
                             "transform": out_transform, "nodata": np.nan, "count": 1, "dtype": 'float32'})
            data = out_image[0].astype('float32')
            mask = (data == src.nodata) | np.isnan(data)
            data[mask] = np.nan
            log_data = np.log1p(data)
            with rasterio.open(cropped_tif, 'w', **out_meta) as dst:
                dst.write(log_data.astype('float32'), 1)
            ntl_crs = src.crs
        gdf_grid_crs = gdf_grid[['grid_id', 'geometry']].to_crs(ntl_crs)
        stats = zonal_stats(gdf_grid_crs, cropped_tif, stats=['mean', 'sum', 'max'], all_touched=False, nodata=np.nan)
        stats_df = pd.DataFrame(stats)
        stats_df.columns = [f'ntl_{col}' for col in stats_df.columns]
        gdf_grid[['ntl_mean', 'ntl_sum', 'ntl_max']] = stats_df.values
        os.remove(cropped_tif); os.rmdir(temp_dir)

    if relief_folder and admin_union_geom:
        print("8. Обработка рельефа")
        from rasterio.merge import merge as rio_merge
        
        tif_files = glob.glob(os.path.join(relief_folder, '*.tif'))
        if not tif_files:
            print("   ⚠️ В папке рельефа не найдено .tif файлов")
        else:
            temp_dir = tempfile.mkdtemp()
            merged_tif = os.path.join(temp_dir, 'relief_merged.tif')
            clipped_tif = os.path.join(temp_dir, 'relief_clipped.tif')
            
            with rasterio.open(tif_files[0]) as src:
                base_crs = src.crs
            
            mosaic, out_transform = rio_merge(tif_files)
            
            with rasterio.open(
                merged_tif, 'w',
                driver='GTiff',
                height=mosaic.shape[1],
                width=mosaic.shape[2],
                count=1,
                dtype=mosaic.dtype,
                crs=base_crs,
                transform=out_transform,
                nodata=np.nan
            ) as dst:
                dst.write(mosaic[0], 1)
            
            with rasterio.open(merged_tif) as src:
                relief_crs = src.crs
                transformer = Transformer.from_crs(gdf_grid.crs, src.crs, always_xy=True)
                admin_geom_transformed = transform(transformer.transform, admin_union_geom)
                
                out_image, out_transform = raster_mask(src, [admin_geom_transformed], crop=True, nodata=np.nan)
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform,
                    "nodata": np.nan,
                    "count": 1,
                    "dtype": 'float32'
                })
                
                with rasterio.open(clipped_tif, 'w', **out_meta) as dst:
                    dst.write(out_image[0].astype('float32'), 1)
            
            gdf_grid_rel = gdf_grid[['grid_id', 'geometry']].to_crs(relief_crs)
            stats = zonal_stats(gdf_grid_rel, clipped_tif, stats=['mean', 'min', 'max', 'std'], 
                                all_touched=False, nodata=np.nan)
            stats_df = pd.DataFrame(stats)
            
            gdf_grid['height_mean'] = stats_df['mean']
            gdf_grid['height_min'] = stats_df['min']
            gdf_grid['height_max'] = stats_df['max']
            gdf_grid['height_std'] = stats_df['std']
            gdf_grid['height_range'] = gdf_grid['height_max'] - gdf_grid['height_min']
            
            os.remove(merged_tif); os.remove(clipped_tif); os.rmdir(temp_dir)

    
    if 'municipality' in gdf_grid.columns and population_df is not None:
        print("9. Распределение населения")
        pop_df = population_df.copy()
        n_cols = pop_df.shape[1]
        pop_df['name'] = pop_df.iloc[:, 1] if n_cols > 1 else pd.Series(dtype=str)
        pop_df['total_population'] = pd.to_numeric(pop_df.iloc[:, 2], errors='coerce').fillna(0) if n_cols > 2 else 0.0
        pop_df['urban_population'] = pd.to_numeric(pop_df.iloc[:, 3], errors='coerce').fillna(0) if n_cols > 3 else 0.0
        pop_df['rural_population'] = pd.to_numeric(pop_df.iloc[:, 4], errors='coerce').fillna(0) if n_cols > 4 else 0.0
        pop_df['name_clean'] = pop_df['name'].astype(str).apply(clean_municipality_name)
        pop_df = pop_df.groupby('name_clean', as_index=False).agg({
            'name': 'first', 'total_population': 'sum',
            'urban_population': 'sum', 'rural_population': 'sum'
        })
        pop_dict = pop_df.set_index('name_clean').to_dict('index')

        print("\nТехническая таблица соответствия названий:")
        mapping_rows = []
        for muni in gdf_grid['municipality'].dropna().unique():
            status = "Отл" if muni in pop_dict else "Не отл"
            found = pop_dict[muni]['name'] if muni in pop_dict else "НЕ НАЙДЕНО"
            print(f"{status} '{muni}' -> '{found}'")
            mapping_rows.append({'municipality_in_grid': muni, 'matched_name': found, 'status': status})
        pd.DataFrame(mapping_rows).to_csv(output.replace('.gpkg', '_mapping.csv'), index=False, encoding='utf-8-sig')

        gdf_grid['total_population'] = 0.0
        gdf_grid['urban_population'] = 0.0
        gdf_grid['rural_population'] = 0.0
        for muni in gdf_grid['municipality'].dropna().unique():
            if muni not in pop_dict: continue
            pop_data = pop_dict[muni]
            mask = gdf_grid['municipality'] == muni
            total_b_area = gdf_grid.loc[mask, 'total_building_area'].sum()
            if total_b_area > 0:
                ratio = gdf_grid.loc[mask, 'total_building_area'] / total_b_area
                gdf_grid.loc[mask, 'total_population'] = pop_data['total_population'] * ratio
                gdf_grid.loc[mask, 'urban_population'] = pop_data['urban_population'] * ratio
                gdf_grid.loc[mask, 'rural_population'] = pop_data['rural_population'] * ratio
            else:
                count = mask.sum()
                if count > 0:
                    gdf_grid.loc[mask, 'total_population'] = pop_data['total_population'] / count
                    gdf_grid.loc[mask, 'urban_population'] = pop_data['urban_population'] / count
                    gdf_grid.loc[mask, 'rural_population'] = pop_data['rural_population'] / count

    print("10. Сохранение")
    float_cols = ['total_building_area', 'building_coverage', 'road_length', 'road_density',
                  'nature_area', 'nature_density', 'municipality_area', 'total_population',
                  'urban_population', 'rural_population', 'ntl_mean', 'ntl_sum', 'ntl_max',
                  'height_mean', 'height_min', 'height_max', 'height_std', 'height_range']
    for c in float_cols:
        if c not in gdf_grid.columns: gdf_grid[c] = 0.0
        gdf_grid[c] = gdf_grid[c].fillna(0.0)
    if 'number_of_buildings' not in gdf_grid.columns: gdf_grid['number_of_buildings'] = 0
    gdf_grid['number_of_buildings'] = gdf_grid['number_of_buildings'].fillna(0).astype(int)

    gdf_grid = gpd.GeoDataFrame(gdf_grid, geometry=gdf_grid.geometry, crs=target_crs)
    os.makedirs(os.path.dirname(os.path.abspath(output)), exist_ok=True)
    gdf_grid.to_file(output, driver='GPKG')
    return output, target_crs

In [ ]:
# Пути к данным
RELIEF_FOLDER = r'D:\project\Рельеф'
NTL_PATH = r'D:\project\Ночные огни.tif'
POP_CSV = r'D:\project\население.csv'
super_dict = {
    'Орел': {
        'admin': r'D:\project\Орел\Админ.gpkg',
        'buildings': r'D:\project\Орел\Здания.gpkg',
        'roads': r'D:\project\Орел\Дороги.gpkg',
        'nature': r'D:\project\Орел\Природа.gpkg'
    },
    'брянск': {
        'admin': r'D:\project\брянск\Админ.gpkg',
        'buildings': r'D:\project\брянск\Здания.gpkg',
        'roads': r'D:\project\брянск\Дороги.gpkg',
        'nature': r'D:\project\брянск\Природа.gpkg'
    },
    'Курск': {
        'admin': r'D:\project\Курск\Админ.gpkg',
        'buildings': r'D:\project\Курск\Здания.gpkg',
        'roads': r'D:\project\Курск\Дороги.gpkg',
        'nature': r'D:\project\Курск\Природа.gpkg'
    }
}

population_data = pd.read_csv(POP_CSV, header=None, sep=';')
output_dir = r'D:\project\final_grids'
os.makedirs(output_dir, exist_ok=True)
processed_paths = []
locked_crs = None

for region_name, layers in super_dict.items():
    out_path = os.path.join(output_dir, f'grid_1km_{region_name}_v11.gpkg')
    path, crs = advanced_grid_pipeline_v11(
        layers, population_data, out_path,
        size_m=1000, target_crs=locked_crs,
        ntl_path=NTL_PATH,
        relief_folder=RELIEF_FOLDER 
    )
    processed_paths.append(path)
    if locked_crs is None: locked_crs = crs

# Слияние 
gdfs_all = [gpd.read_file(p) for p in processed_paths]
gdf_combined = pd.concat(gdfs_all, ignore_index=True)
gdf_combined = gdf_combined.sort_values('municipality_area', ascending=False, na_position='last')
gdf_combined = gdf_combined.drop_duplicates(subset='geometry', keep='first')
gdf_combined = gdf_combined.reset_index(drop=True)
gdf_combined['grid_id'] = range(len(gdf_combined))

combined_path = os.path.join(output_dir, 'grid_1km_Combined_v11.gpkg')
gdf_combined.to_file(combined_path, driver='GPKG')
